# 3052. Maximize Items

## Problem Description
Leetcode warehouse wants to maximize the number of items it can stock in a **500,000 square feet warehouse**.  

- The warehouse should stock as many **prime_eligible** items as possible first.  
- After filling with prime items, use the remaining square footage to stock the maximum number of **not_prime** items.  
- Item counts must be whole numbers (integers).  
- If the count for the not_prime category is 0, output 0 for that category.  
- Return the result table ordered by item count in **descending order**.  

---

## Schema

### Table: Inventory
| Column Name    | Type     | Description                          |
|----------------|----------|--------------------------------------|
| item_id        | INT      | Unique identifier for each item       |
| item_type      | VARCHAR  | Type of item (`prime_eligible` or `not_prime`) |
| item_category  | VARCHAR  | Category of the item (e.g., Watches, Shoes) |
| square_footage | DECIMAL  | Space occupied by the item in sq. ft. |

**Primary Key:** `item_id`

---

## Sample Data

### Inventory
| item_id | item_type      | item_category | square_footage |
|---------|----------------|---------------|----------------|
| 1374    | prime_eligible | Watches       | 68.00          |
| 4245    | not_prime      | Art           | 26.40          |
| 5743    | prime_eligible | Software      | 325.00         |
| 8543    | not_prime      | Clothing      | 64.50          |
| 2556    | not_prime      | Shoes         | 15.00          |
| 2452    | prime_eligible | Scientific    | 85.00          |
| 3255    | not_prime      | Furniture     | 22.60          |
| 1672    | prime_eligible | Beauty        | 8.50           |
| 4256    | prime_eligible | Furniture     | 55.50          |
| 6325    | prime_eligible | Food          | 13.20          |

---

## Expected Output

| item_type      | item_count |
|----------------|------------|
| prime_eligible | 5400       |
| not_prime      | 8          |

### Explanation
- **Prime Eligible Items:**  
  - 6 items total, combined square footage = 555.20.  
  - Warehouse can fit 900 sets of these items → 5400 items total.  
  - Space used = 499,680 sq. ft.  

- **Not Prime Items:**  
  - 4 items total, combined square footage = 128.50.  
  - Remaining space = 320 sq. ft.  
  - Warehouse can fit 2 sets of these items → 8 items total.  

---

## PySpark Code: Create DataFrame and Temp View

```python


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType
from decimal import Decimal

# Schema for Inventory
inventory_schema = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("item_type", StringType(), False),
    StructField("item_category", StringType(), False),
    StructField("square_footage", DecimalType(10,2), False)
])

# Data for Inventory
inventory_data = [
    (1374, "prime_eligible", "Watches", Decimal("68.00")),
    (4245, "not_prime", "Art", Decimal("26.40")),
    (5743, "prime_eligible", "Software", Decimal("325.00")),
    (8543, "not_prime", "Clothing", Decimal("64.50")),
    (2556, "not_prime", "Shoes", Decimal("15.00")),
    (2452, "prime_eligible", "Scientific", Decimal("85.00")),
    (3255, "not_prime", "Furniture", Decimal("22.60")),
    (1672, "prime_eligible", "Beauty", Decimal("8.50")),
    (4256, "prime_eligible", "Furniture", Decimal("55.50")),
    (6325, "prime_eligible", "Food", Decimal("13.20"))
]

# Create DataFrame
inventory_df = spark.createDataFrame(inventory_data, inventory_schema)

# Register Temp View
inventory_df.createOrReplaceTempView("Inventory")

# Quick check
inventory_df.show()


In [0]:
%sql
--% sql
WITH cte AS (
		SELECT DISTINCT item_type,
			count(item_type) OVER (PARTITION BY item_type) AS count_item_type,
			sum(square_footage) OVER (PARTITION BY item_type) AS sq_ft_per_item_type,
			floor(500000 / sum(square_footage) OVER (PARTITION BY item_type)) AS eligible_item_type_PER_STORAGE,
			count(item_type) OVER (PARTITION BY item_type) * floor(500000 / sum(square_footage) OVER (PARTITION BY item_type)) AS item_storage_capacity_count,
	
  floor(500000 / sum(square_footage) OVER (PARTITION BY item_type))
  *
  	sum(square_footage) OVER (PARTITION BY item_type) 
  
  
  
   AS total_space_occupied
		FROM Inventory
		)

select item_type , eligible_item_type_PER_STORAGE as  item_count from cte where item_type = 'prime_eligible'

union
select item_type  ,floor(
 (500000 - (select total_space_occupied from cte where item_type = 'prime_eligible') )/
(select sq_ft_per_item_type from cte where item_type = 'not_prime' ) )as item_count  from cte 
where item_type ='not_prime'





---

```markdown
# Documentation: Maximize Items in Warehouse (500,000 sq. ft.)

This query calculates how many **prime_eligible** and **not_prime** items can be stored in a warehouse with a fixed capacity of 500,000 square feet.

---

## Step 1: Build the Base CTE
```sql
WITH cte AS (
    SELECT DISTINCT 
        item_type,

        -- Total number of items of this type
        COUNT(item_type) OVER (PARTITION BY item_type) AS total_items_per_type,

        -- Combined square footage of all items of this type
        SUM(square_footage) OVER (PARTITION BY item_type) AS total_sqft_per_type,

        -- How many full sets of this item type can fit in 500,000 sq. ft.
        FLOOR(500000 / SUM(square_footage) OVER (PARTITION BY item_type)) AS max_sets_per_type,

        -- Total items that can be stored = items per set * number of sets
        COUNT(item_type) OVER (PARTITION BY item_type) 
            * FLOOR(500000 / SUM(square_footage) OVER (PARTITION BY item_type)) AS max_items_per_type,

        -- Actual space occupied by those sets
        FLOOR(500000 / SUM(square_footage) OVER (PARTITION BY item_type)) 
            * SUM(square_footage) OVER (PARTITION BY item_type) AS total_space_occupied
    FROM Inventory
)
```

### Explanation
- **`COUNT(item_type)`** → how many distinct items exist for each type (`prime_eligible` or `not_prime`).  
- **`SUM(square_footage)`** → total space required for one full set of all items of that type.  
- **`FLOOR(500000 / SUM(...))`** → how many complete sets fit in the warehouse.  
- **`COUNT * FLOOR(...)`** → total number of items that can be stored.  
- **`total_space_occupied`** → actual square footage used by those sets.

---

## Step 2: Calculate Prime Items
```sql
SELECT 
    item_type, 
    max_sets_per_type AS item_count
FROM cte 
WHERE item_type = 'prime_eligible'
```

- For prime items, we simply take the maximum number of sets that fit in the warehouse.  
- This ensures we maximize prime storage first.

---

## Step 3: Calculate Non‑Prime Items
```sql
UNION
SELECT 
    item_type,
    FLOOR(
        (500000 - (SELECT total_space_occupied 
                   FROM cte WHERE item_type = 'prime_eligible'))
        / (SELECT total_sqft_per_type 
           FROM cte WHERE item_type = 'not_prime')
    ) AS item_count
FROM cte 
WHERE item_type = 'not_prime'
```

- After prime items are placed, calculate **remaining space**:  
  `500000 - total_space_occupied_by_prime`.  
- Divide that remaining space by the total square footage required for one set of non‑prime items.  
- Use `FLOOR` to ensure we only count complete sets.  
- This gives the maximum number of non‑prime items that can fit.

---

## Suggested Column Names / Aliases
To make the query easier to read:
- `count_item_type` → `total_items_per_type`  
- `sq_ft_per_item_type` → `total_sqft_per_type`  
- `eligible_item_type_PER_STORAGE` → `max_sets_per_type`  
- `item_storage_capacity_count` → `max_items_per_type`  
- `total_space_occupied` → `space_used_by_sets`  

---

## Final Output
- Two rows: one for `prime_eligible`, one for `not_prime`.  
- Each row shows the maximum number of items that can be stored in the warehouse.  
- Ordered by item count (you can add `ORDER BY item_count DESC` at the end if needed).

---
```



In [0]:
%sql
WITH cte AS (
		SELECT DISTINCT item_type,
			count(item_type) OVER (PARTITION BY item_type) AS total_items_per_type,
			sum(square_footage) OVER (PARTITION BY item_type) AS total_sqft_per_type,
			floor(500000 / sum(square_footage) OVER (PARTITION BY item_type)) AS max_sets_per_type
		FROM Inventory
		),
	cte2 AS (
		SELECT item_type,
			total_items_per_type,
			total_sqft_per_type,
			max_sets_per_type,
			(total_sqft_per_type * max_sets_per_type) AS space_used_by_sets
		FROM cte
		)

SELECT item_type,
	max_sets_per_type AS item_count
FROM cte2
WHERE item_type = 'prime_eligible'

UNION

SELECT item_type,
	floor((
			500000 - (
				SELECT space_used_by_sets
				FROM cte2
				WHERE item_type = 'prime_eligible'
				)
			) / total_sqft_per_type) AS item_count
FROM cte2
WHERE item_type = 'not_prime'
